# Train Classifier

## 1. Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_TRAIN = DATA_PROCESSED / "train.csv"
SPLIT_VAL = DATA_PROCESSED / "val.csv"

MODEL_KEY = "vit"

MODEL_CONFIGS = {
    "cnn_baseline":  {"timm_name": "resnet50",              "description": "Baseline CNN (ResNet-50)"},
    "attention_cnn": {"timm_name": "resnet50",               "description": "ResNet-50 + CBAM attention"},
    #"vit":           {"timm_name": "vit_base_patch16_224",   "description": "Vision Transformer B/16"},
    "vit":           {"timm_name": "deit_small_patch16_224",   "description": "Data-efficient Transformer (DeiT-Small)"},
}

IMAGE_SIZE = 224
BATCH_SIZE = 8          
NUM_WORKERS = 0         
LEARNING_RATE = 1e-5 if MODEL_KEY == "vit" else 2e-5
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 10
FREEZE_EPOCHS = 3           
HEAD_DROPOUT = 0.5
RANDOM_SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Training: {MODEL_KEY} ({MODEL_CONFIGS[MODEL_KEY]['description']})")


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seeds set to {seed} (random, numpy, torch, cuda; cuDNN deterministic)")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(RANDOM_SEED)

## 2. GPU check 

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
else:
    print("WARNING: no GPU detected.")

## 3. Dataset & augmentation

In [ ]:
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

def build_transforms(train: bool, strong: bool = False):
    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if not train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(), normalize,
        ])
    if strong:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(), normalize,
        ])
    return transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
        transforms.ToTensor(), normalize,
    ])


class SkinLesionDataset(Dataset):
    def __init__(self, csv_path, train: bool, minority_strong_aug: bool = True):
        self.df = pd.read_csv(csv_path)
        self.train = train
        self.minority_strong_aug = minority_strong_aug and train
        self.tf_normal = build_transforms(train, strong=False)
        self.tf_strong = build_transforms(train, strong=True) if self.minority_strong_aug else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])
        img = self.tf_strong(img) if (self.minority_strong_aug and label == 1) else self.tf_normal(img)
        return img, torch.tensor(label, dtype=torch.long)

    @property
    def labels(self):
        return self.df["label"].values


train_ds = SkinLesionDataset(SPLIT_TRAIN, train=True, minority_strong_aug=True)
val_ds = SkinLesionDataset(SPLIT_VAL, train=False)
print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")


## 4. Class imbalance handling

In [ ]:
import numpy as np

def compute_class_weights(y_train, num_classes=2, scheme="balanced"):
    y = np.asarray(y_train)
    counts = np.bincount(y, minlength=num_classes).astype(float)
    if scheme == "balanced":
        w = len(y) / (num_classes * counts)
    elif scheme == "sqrt_inv":
        w = 1.0 / np.sqrt(counts); w = w / w.sum() * num_classes
    else:
        w = 1.0 / counts; w = w / w.sum() * num_classes
    return torch.tensor(w, dtype=torch.float)

y_train = train_ds.labels
counts = np.bincount(y_train, minlength=2)
print("Training class distribution:")
print(f"  non-melanoma: {counts[0]} ({100*counts[0]/len(y_train):.1f}%)")
print(f"  melanoma:     {counts[1]} ({100*counts[1]/len(y_train):.1f}%)")

class_weights = compute_class_weights(y_train, scheme="balanced").to(device)
print(f"\nClass weights (balanced): {class_weights.tolist()}")
print("If training is unstable, try scheme='sqrt_inv' for a gentler weighting.")


## 5. Model definition

In [ ]:
import timm
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class AttentionCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.cbam = CBAM(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feats = self.cbam(self.backbone(x))
        return self.fc(self.dropout(self.pool(feats).flatten(1)))


def build_model(model_key, num_classes=2, pretrained=True, dropout=0.3):
    timm_name = MODEL_CONFIGS[model_key]["timm_name"]
    if model_key == "attention_cnn":
        return AttentionCNN(timm_name, num_classes, pretrained)
    return timm.create_model(timm_name, pretrained=pretrained,
                             num_classes=num_classes, drop_rate=dropout)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = build_model(MODEL_KEY, dropout=HEAD_DROPOUT).to(device)

HEAD_KEYS = ("fc.", "head.", "classifier.", "cbam.")  

def set_backbone_frozen(model, frozen: bool):
    for name, p in model.named_parameters():
        is_new = any(k in name for k in HEAD_KEYS)
        p.requires_grad = True if is_new else (not frozen)

set_backbone_frozen(model, frozen=True)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 (backbone frozen): {n_trainable:,} trainable params")

print(f"Model: {MODEL_KEY}")
print(f"Trainable parameters: {count_parameters(model):,}")

## 6. Training loop

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score,
                             precision_score, f1_score, confusion_matrix, roc_auc_score)
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
import time, csv

g = torch.Generator()
g.manual_seed(RANDOM_SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"),
                          worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
use_amp = (device.type == "cuda")
scaler = GradScaler(enabled=use_amp)


def melanoma_report(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity_melanoma": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "precision_melanoma": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_melanoma": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


def run_epoch(loader, train_mode: bool):
    model.train() if train_mode else model.eval()
    total_loss, n_seen = 0.0, 0
    all_true, all_pred, all_prob = [], [], []
    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            with autocast(device_type="cuda", enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, labels)
            if train_mode:
                if use_amp:
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            probs = torch.softmax(logits.float(), dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            total_loss += loss.item() * labels.size(0)
            n_seen += labels.size(0)
            all_true.append(labels.detach().cpu().numpy())
            all_pred.append(preds.detach().cpu().numpy())
            all_prob.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(all_true); y_pred = np.concatenate(all_pred); y_prob = np.concatenate(all_prob)
    m = melanoma_report(y_true, y_pred)
    m["loss"] = total_loss / max(n_seen, 1)
    m["auc"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan")
    return m


history_path = RESULTS_DIR / f"{MODEL_KEY}_history.csv"
ckpt_path = MODELS_DIR / f"{MODEL_KEY}_best.pth"
fields = ["epoch", "split", "loss", "accuracy", "balanced_accuracy",
          "sensitivity_melanoma", "specificity", "precision_melanoma", "f1_melanoma", "auc"]

history = []
best_score, patience_left = -1.0, EARLY_STOPPING_PATIENCE

with open(history_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        print(f"Epoch {epoch}/{NUM_EPOCHS} - training...", flush=True)

        if epoch == FREEZE_EPOCHS + 1:
            set_backbone_frozen(model, frozen=False)
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
            )
            n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  -> Unfroze backbone: now {n_trainable:,} trainable params", flush=True)

        tr = run_epoch(train_loader, train_mode=True)
        va = run_epoch(val_loader, train_mode=False)
        scheduler.step()

        writer.writerow({"epoch": epoch, "split": "train", **tr})
        writer.writerow({"epoch": epoch, "split": "val", **va})
        f.flush()
        history.append((epoch, tr, va))

        print(f"Epoch {epoch:>2}/{NUM_EPOCHS}  ({time.time()-t0:.0f}s)")
        print(f"  train  loss={tr['loss']:.4f}  sens={tr['sensitivity_melanoma']:.3f}  bal_acc={tr['balanced_accuracy']:.3f}")
        print(f"  val    loss={va['loss']:.4f}  sens={va['sensitivity_melanoma']:.3f}  bal_acc={va['balanced_accuracy']:.3f}  auc={va['auc']:.3f}")

        score = va["balanced_accuracy"]
        if score > best_score:
            best_score, patience_left = score, EARLY_STOPPING_PATIENCE
            torch.save({"model_state": model.state_dict(), "model_key": MODEL_KEY,
                       "epoch": epoch, "val_metrics": va}, ckpt_path)
            print(f"  -> new best (bal_acc={score:.3f}), checkpoint saved")
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"\nEarly stopping at epoch {epoch}.")
                break
        print()

print(f"Done. Best val balanced accuracy: {best_score:.3f}")
print(f"History: {history_path}\nCheckpoint: {ckpt_path}")


## 7. Learning curves

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.read_csv(history_path)
train_h = hist_df[hist_df.split == "train"]
val_h = hist_df[hist_df.split == "val"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_h.epoch, train_h.loss, label="train")
axes[0].plot(val_h.epoch, val_h.loss, label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(train_h.epoch, train_h.balanced_accuracy, label="train")
axes[1].plot(val_h.epoch, val_h.balanced_accuracy, label="val")
axes[1].set_title("Balanced accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(train_h.epoch, train_h.sensitivity_melanoma, label="train")
axes[2].plot(val_h.epoch, val_h.sensitivity_melanoma, label="val")
axes[2].set_title("Melanoma sensitivity (recall)"); axes[2].set_xlabel("Epoch"); axes[2].legend()

fig.suptitle(f"{MODEL_KEY} — training curves")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{MODEL_KEY}_curves.png", dpi=150)
plt.show()
